<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_33_django_intro/note_lesson_33_django.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Урок 33 — Django: модель, ORM, view і адмінка

| Крок | Що робимо |
|---|---|
| 0 | запускаємо Django-проєкт `hello_project` у ноутбуці |
| 1 | створення записів (вправа 1) |
| 2 | `filter`, `order_by`, `values_list` (вправи 2–3) |
| 3 | `get` і його винятки (вправа 4) |
| 4 | `update` і `.query` (вправи 5–6) |
| 5 | view і шаблон через тестовий клієнт (вправа 7) |
| 6 | адмінка (вправа 8) |
| 7 | знайди помилку (вправа 9) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія — у книзі курсу: [Урок 33](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_33/); поглиблено — у [Django-книзі викладача](https://nikoriakviktot.github.io/notes_chat_app/) (кроки 1–2 маршруту Zero to Hero).

---
## 0. Проєкт у ноутбуці

Готовий проєкт уроку — папка `hello_project` поруч із ноутбуком (модель `Note` з полями `title`, `content`, `is_pinned`, `priority`, `created_at`). У Colab ноутбук відкривається без файлів репозиторію — клітинка нижче завантажить лише папку уроку (`git clone --sparse`) і встановить Django.

Зазвичай з Django працюють у терміналі (`python manage.py shell`, `runserver`). У ноутбуці ми робимо те саме з Python: `django.setup()` завантажує налаштування й моделі.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_33_django_intro"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("hello_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

if importlib.util.find_spec("django") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "Django>=5.2,<6"], check=True)

os.chdir("hello_project")
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-3:]))

In [ ]:
import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
# Jupyter виконує клітинки всередині asyncio-циклу, а ORM Django синхронний і за замовчуванням
# це забороняє. Для ноутбука дозволяємо явно (так радить документація Django).
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from django.contrib.auth.models import User
from hello_app.models import Note

Note.objects.all().delete()            # кожен запуск ноутбука — з порожньої таблиці
print("Django", django.get_version(), "| нотаток:", Note.objects.count())

---
## 1. Створення записів

`Note.objects.create(...)` — створити й одразу зберегти. `Note(...)` + `.save()` — те саме у два кроки.

### Вправа 1

Створи три нотатки:

| title | content | is_pinned |
|---|---|---|
| `Купити молоко` | `2 л` | ні |
| `Django ORM` | `filter, get` | так |
| `Django admin` | *(порожньо)* | ні |

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
Note.objects.create(title="Купити молоко", content="2 л")
Note.objects.create(title="Django ORM", content="filter, get", is_pinned=True)
Note.objects.create(title="Django admin")
# END SOLUTION

assert Note.objects.count() == 3
assert Note.objects.get(title="Django ORM").is_pinned
print([str(note) for note in Note.objects.all()])
print("✅ Вправа 1 пройдена")

**🔮 Прогноз:** У якому порядку `Note.objects.all()` повернув нотатки і чому «Django ORM» перша?

<details>
<summary>Відповідь</summary>

`Meta.ordering = ["-is_pinned", "-created_at"]`: спершу закріплені, потім від найновішої. «Django ORM» закріплена, далі «Django admin» (створена останньою), потім «Купити молоко».

</details>

---
## 2. filter, order_by, values_list

### Вправа 2

`django_titles` — список заголовків нотаток, у заголовку яких є `django` (без урахування регістру), **за алфавітом**. Один QuerySet: `filter` + `order_by` + `values_list(..., flat=True)`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
django_titles = list(Note.objects.filter(title__icontains="django").order_by("title").values_list("title", flat=True))
# END SOLUTION

assert django_titles == ["Django ORM", "Django admin"], django_titles
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Чому `"Django ORM"` раніше за `"Django admin"` в алфавітному порядку?

<details>
<summary>Відповідь</summary>

SQLite порівнює рядки за кодами символів: великі латинські літери (`O` = 79) йдуть раніше за малі (`a` = 97). Порядок «як у словнику» залежить від collation бази; у PostgreSQL з українською локаллю він буде іншим.

</details>

### Вправа 3

`empty_count` — скільки нотаток **без тексту**; `with_text` — заголовки нотаток **з текстом** (`exclude`), порядок — як за замовчуванням.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
empty_count = Note.objects.filter(content="").count()
with_text = [note.title for note in Note.objects.exclude(content="")]
# END SOLUTION

assert empty_count == 1
assert with_text == ["Django ORM", "Купити молоко"], with_text
print("✅ Вправа 3 пройдена")

---
## 3. get і його винятки

`get()` повертає рівно один об'єкт, інакше — `Note.DoesNotExist` або `Note.MultipleObjectsReturned`.

### Вправа 4

`note_or_none(pk)` повертає нотатку з таким `id` або `None`, якщо її немає.

In [ ]:
def note_or_none(pk):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        return Note.objects.get(pk=pk)
    except Note.DoesNotExist:
        return None
    # END SOLUTION


first = Note.objects.order_by("id").first()
assert note_or_none(first.pk) == first
assert note_or_none(10**6) is None
print("✅ Вправа 4 пройдена")

---
## 4. update і .query

### Вправа 5

Одним запитом `update` закріпи **всі** нотатки, в заголовку яких є `django`. Кількість оновлених рядків — у `updated`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
updated = Note.objects.filter(title__icontains="django").update(is_pinned=True)
# END SOLUTION

assert updated == 2
assert Note.objects.filter(is_pinned=True).count() == 2
print("✅ Вправа 5 пройдена")

### Вправа 6

Побудуй QuerySet `urgent` — закріплені нотатки з пріоритетом **не нижче 2** — і подивись його SQL через `urgent.query`. Збережи SQL рядком у `sql`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
urgent = Note.objects.filter(is_pinned=True, priority__gte=2)
sql = str(urgent.query)
# END SOLUTION

print(sql)
assert '"priority" >= 2' in sql and '"is_pinned"' in sql and "ORDER BY" in sql
assert urgent.count() == 2
print("✅ Вправа 6 пройдена")

---
## 5. View і шаблон

Сервер `runserver` у ноутбуці не запускаємо: Django має **тестовий клієнт**, який передає запит у view без мережі — так само, як браузер через `runserver`.

### Вправа 7

Зроби `GET /notes/` тестовим клієнтом. Збережи відповідь у `response`.

In [ ]:
from django.test import Client
from django.test.utils import setup_test_environment

setup_test_environment()               # дозволяє хост testserver і збирає використані шаблони
client = Client()
# YOUR CODE HERE
# BEGIN SOLUTION
response = client.get("/notes/")
# END SOLUTION

html = response.content.decode()
assert response.status_code == 200
assert "<h1>Нотатки (3)</h1>" in html
assert html.index("Django ORM") < html.index("Купити молоко"), "закріплені — першими"
print(html[:400])
print("✅ Вправа 7 пройдена")

---
## 6. Адмінка

### Вправа 8

1. Отримай `/admin/hello_app/note/` **без входу** — код відповіді в `anonymous_code`.
2. Створи (якщо ще немає) суперкористувача `teacher` з паролем `lesson-33` і увійди клієнтом: `client.login(...)`.
3. Отримай ту саму сторінку ще раз — відповідь у `admin_page`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
anonymous_code = client.get("/admin/hello_app/note/").status_code
if not User.objects.filter(username="teacher").exists():
    User.objects.create_superuser("teacher", "teacher@example.com", "lesson-33")
client.login(username="teacher", password="lesson-33")
admin_page = client.get("/admin/hello_app/note/")
# END SOLUTION

assert anonymous_code == 302, "без входу — перенаправлення на логін"
assert admin_page.status_code == 200
assert "Купити молоко" in admin_page.content.decode()
print("✅ Вправа 8 пройдена")

**🔮 Прогноз:** Що лежить у полі `User.objects.get(username='teacher').password` — сам пароль?

<details>
<summary>Відповідь</summary>

Ні: рядок `pbkdf2_sha256$<раунди>$<сіль>$<хеш>`. Django зберігає лише хеш, а `check_password` порівнює хеші (урок 16, урок 40).

</details>

---
## 7. Знайди помилку

```python
def first_unpinned_title():
    return Note.objects.get(is_pinned=False).title
```

Сьогодні працює. Але щойно незакріплених нотаток стане дві — впаде, а коли не стане жодної — теж.

### Вправа 9

Перепиши `first_unpinned_title()`: повертає заголовок **найновішої** незакріпленої нотатки або `None`, якщо таких немає. Без `try/except`.

In [ ]:
def first_unpinned_title():
    # YOUR CODE HERE
    # BEGIN SOLUTION
    note = Note.objects.filter(is_pinned=False).order_by("-created_at").first()
    return note.title if note else None
    # END SOLUTION


assert first_unpinned_title() == "Купити молоко"
Note.objects.create(title="Ще одна")
assert first_unpinned_title() == "Ще одна"
Note.objects.filter(is_pinned=False).update(is_pinned=True)
assert first_unpinned_title() is None
print("✅ Вправа 9 пройдена")

---
## Самоперевірка

1. Що робить `makemigrations`, а що — `migrate`?
2. Чим `filter()` відрізняється від `get()`?
3. Коли QuerySet іде в базу?
4. Чим `note.save()` відрізняється від `QuerySet.update()`?
5. Навіщо `app_name` і `name` у маршрутах?

<details>
<summary>Відповіді</summary>

1. `makemigrations` створює файл міграції з опису моделей; `migrate` виконує міграції на базі.
2. `filter()` — QuerySet з 0…N записів; `get()` — рівно один об'єкт або виняток `DoesNotExist` / `MultipleObjectsReturned`.
3. Коли потрібні дані: цикл, `list()`, `len()`, `print()`, індекс, `first()`, `count()`, `exists()`.
4. `save()` зберігає один завантажений об'єкт; `update()` — один SQL `UPDATE` для всіх рядків QuerySet.
5. Щоб посилатися на маршрут за ім'ям (`reverse("hello_app:note_list")`, `{% url %}`), а не за жорстко прописаною адресою.

</details>

## Далі

- Книга курсу: [Урок 33](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_33/) — створення проєкту з нуля, міграції та їхній SQL, адмінка зі скріншотами.
- Django-книга: [крок 1](https://nikoriakviktot.github.io/notes_chat_app/tutorials/01_hello_django/) і [крок 2](https://nikoriakviktot.github.io/notes_chat_app/tutorials/02_first_model/) маршруту Zero to Hero.
- Запусти проєкт у терміналі: `python manage.py runserver` у папці `hello_project` і відкрий `/notes/` та `/admin/`.
- **Урок 34** — форми, base-шаблон, Bootstrap і CRUD нотаток.